# AIC365 Natural Language Processing — Lab 4
## FA23-BAI-006 | Text Normalization, Cleaning & Minimum Edit Distance

**Scope:** Stage v (Verify) — Home Activities only.

The manual refers to a `support_chats` dataset, but that dataset was not attached with the lab manual. The notebook includes a small illustrative set of support chats and a vocabulary from the task examples. Replace the sample messages with the official dataset when provided.

### Setup

In [ ]:
import re
import string
from collections import Counter
import pandas as pd

support_chats = [
    "my pakage has not arived yet please chek the status",
    "i recieved the wrong item and need a refund",
    "i beleive the app is crashin when i open it",
    "the delievery was late and the box was damaged",
    "can you chek why my order has not arrived",
    "i definately paid but the payment is not showing",
    "the product is good but the pakage was broken",
    "wich address will the replacement be sent to",
    "the item occured with a scratch and i need help"
]

vocab_list = [
    "package", "arrived", "check", "receive", "believe", "definitely",
    "crashing", "delivery", "which", "occurred", "order", "payment",
    "product", "address", "replacement", "refund", "status", "broken",
    "item", "help", "late", "damaged", "showing", "sent", "wrong"
]

### Activity 1 — Clean chats end to end
Lowercase, remove stop words, lemmatize, and report vocabulary size before and after cleaning. The simple rule-based lemmatizer is a lightweight fallback so the notebook runs without downloading a spaCy model.

In [ ]:
stop_words = set(the a an and or but is are was were has have had i my it when to of for not yet please can you why with this that be will on after.split())

def simple_lemma(word):
    # Small, transparent rule-based fallback; use spaCy lemmatization for a full NLP pipeline.
    if len(word) > 5 and word.endswith("ing"):
        return word[:-3]
    if len(word) > 4 and word.endswith("ed"):
        return word[:-2]
    if len(word) > 4 and word.endswith("ies"):
        return word[:-3] + "y"
    if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word

def clean_chat(text):
    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    return [simple_lemma(w) for w in words if w not in stop_words]

vocab_before = set(w for chat in support_chats for w in re.findall(r"\b[a-zA-Z]+\b", chat.lower()))
cleaned_chats = [clean_chat(chat) for chat in support_chats]
vocab_after = set(w for row in cleaned_chats for w in row)
reduction = (len(vocab_before) - len(vocab_after)) / len(vocab_before) * 100 if vocab_before else 0

print("Vocabulary before cleaning:", len(vocab_before))
print("Vocabulary after cleaning:", len(vocab_after))
print(f"Vocabulary reduction: {reduction:.2f}%")

### Activity 2 — Compare stemming and lemmatization
Apply Porter stemming and spaCy lemmatization to ten words. If the spaCy English model is unavailable, the notebook reports the simple lemma fallback.

In [ ]:
words_to_compare = list(sorted(vocab_after))[:10]

try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    spacy_available = True
except Exception:
    spacy_available = False

from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

comparison = []
for word in words_to_compare:
    stem = stemmer.stem(word)
    if spacy_available:
        lemma = nlp(word)[0].lemma_
    else:
        lemma = simple_lemma(word)
    comparison.append({
        "word": word,
        "Porter stem": stem,
        "lemma": lemma,
        "stem looks unlike word": stem != word and (stem.endswith(("i", "li", "ss")) or len(stem) < 4)
    })

display(pd.DataFrame(comparison))
if not spacy_available:
    print("Note: spaCy model en_core_web_sm was not available; the displayed lemma uses a simple fallback.")

### Activity 3 — Edit-distance spelling suggestions
Find the closest vocabulary word for each misspelling and display the corrected chats.

In [ ]:
try:
    from nltk import edit_distance
except ImportError:
    def edit_distance(a, b):
        # Dynamic-programming Levenshtein distance fallback
        dp = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            new = [i]
            for j, cb in enumerate(b, 1):
                new.append(min(new[-1] + 1, dp[j] + 1, dp[j-1] + (ca != cb)))
            dp = new
        return dp[-1]

misspellings = ["pakage", "arived", "chek", "beleive", "definately",
                "crashin", "delievery", "wich", "recieved", "occured", "seperate"]

def suggest(word, vocabulary=vocab_list):
    word = word.lower()
    ranked = sorted(((edit_distance(word, v), v) for v in vocabulary))
    return ranked[0][1], ranked[0][0]

suggestion_rows = []
for word in misspellings:
    correction, distance = suggest(word)
    suggestion_rows.append({"misspelled": word, "suggestion": correction, "distance": distance})
display(pd.DataFrame(suggestion_rows))

misspelling_map = {w: suggest(w)[0] for w in misspellings}
def autocorrect_message(message):
    words = re.findall(r"[A-Za-z]+|[^A-Za-z]+", message)
    return "".join(misspelling_map.get(w.lower(), w) if w.isalpha() else w for w in words)

corrected_messages = [autocorrect_message(chat) for chat in support_chats]
display(pd.DataFrame({"original": support_chats, "corrected": corrected_messages}))

### Activity 4 — Autocorrect messages 1, 4, 6 and 9
Show the original and corrected version of the four specified messages (numbered from 1).

In [ ]:
selected_indices = [0, 3, 5, 8]
selected_rows = []
for idx in selected_indices:
    selected_rows.append({
        "message_number": idx + 1,
        "original": support_chats[idx],
        "corrected": autocorrect_message(support_chats[idx])
    })
display(pd.DataFrame(selected_rows))

### Note
The supplied manual names `support_chats` but does not provide its actual contents or the complete vocabulary file in the visible lab material. Results here demonstrate the required workflow on sample messages, not official dataset results.